# Day 48：阶段项目三(上)——多 Agent 智能办公助手（Jupyter Notebook 实战手册）

> **所属阶段**：第 4 阶段 · 第四阶段：Agent 智能体开发与多 Agent 协同 ｜ **第 7 周**：Agent 工程化护栏、行业场景与多 Agent 办公助手实战  
> **GitLab 工单**：`STARMIND-048` ｜ **开发分支**：`feat/day48-project3-office-tools-agents`  
> **配套 HTML5 课件**：`courseware/day48/index.html` ｜ **独立版本快照**：`versions/day48/`

---

## 🎙️ 老陈开场旁白

早啊各位！今天正式开干**阶段项目三（上）——多 Agent 智能办公助手**！这是咱们第四阶段的综合大考，也是写进简历里最硬核的企业级 Agent 项目。需求来自星辰数科全体员工的真实日常：每天要查公司制度、搜行业资讯、约跨部门会议、写对外合作邮件、还要填报销和请假审批单。今天（上集）我们先把架构蓝图定死，亲手写完 **5 大企业办公工具**（日程冲突检测与预定、邮件草拟与校验、行业资讯搜索、内部文档 RAG、费用/请假审批单）以及 **3 位职责隔离的领域专家 Agent**，并给每个高危操作贴好安全风险标签！

### 承上启下
- **昨日基线**：前面九天我们把 ReAct、LangGraph、Checkpointer/HITL、多 Agent Supervisor、MCP、Guardrails 和 Text-to-SQL 全部练熟了。从今天开始，我们要把所有技能融会贯通，打造第四阶段毕业大项目——**星辰智脑「多 Agent 智能办公助手」**！
- **明日预告**：明天（Day 49）进入阶段项目三（下）：把今天建好的 5 大工具和 3 名专家 Agent 装配进带 SQLite Checkpointer + 高危动作 HITL 人工确认的状态图编排器，并封装生产级 FastAPI 服务完成全链路联调测试！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day48'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-048）

星辰数科员工日常办公分散在 OA 审批、企业日历、企业邮箱、Confluence 知识库和外部资讯 5 个孤立系统中，完成一次“查差旅标准 -> 约评审会 -> 发邮件通知 -> 提报销单”的复合任务平均需要切换 4 个网页耗时 25 分钟。

**今日迭代目标：**
- 完成「多 Agent 智能办公助手」整体四层架构设计（API 网关层 -> 编排与 HITL 持久层 -> 专家 Agent 层 -> 企业工具与数据层）。
- 实现 5 个完整可本地运行的企业办公工具：`calendar_tool`、`email_draft_tool`、`web_search_tool`、`doc_rag_tool`、`approval_tool`。
- 实现 3 位领域专家 Agent：`SchedulerAgent`（日程/邮件）、`KnowledgeAgent`（文档 RAG/资讯搜索）、`ApprovalAgent`（审批单），并输出明确的风险等级标记。

**验收标准：**
- 日程工具支持检测会议室/参会人时间冲突，冲突时拒绝预定并给出冲突详情；空闲时段预定成功。
- 邮件工具校验收件人邮箱格式合法性并生成草稿（标记为 `HIGH_RISK`，等待人工确认后才真正外发）。
- 审批工具支持请假与报销单创建，金额 > 5000 元自动标记 `HIGH_RISK` 需总监人工复核；文档 RAG 与搜索工具可精准召回相关条目，所有单测本地 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 1. 为什么必须统一 `ToolResult` 契约？

如果日程工具返回字符串、审批工具返回 dict、邮件工具直接抛异常，上层 Multi-Agent 编排器很快会写满丑陋的 `isinstance` 兼容代码。统一封装 `ToolResult(ok, tool_name, risk_level, data, message)` 后，编排器只需判断 `result.risk_level == 'HIGH_RISK'` 即可通用地触发 HITL 审批断点！

> **⚠️ 老陈避坑**：坑点：不要把业务可预期失败（如“会议时间冲突”“邮箱格式错误”）当成未捕获异常直接 500 炸掉，而应返回 `ToolResult(ok=False, message='...')`，让 Agent 看懂原因并主动换时间或向用户追问。

### 2. 2. 经典时间区间重叠判定公式

判断两个时间段 `[start1, end1)` 与 `[start2, end2)` 是否冲突，最简洁无 Bug 的数学条件是：`start1 < end2 and start2 < end1`（注意使用半开区间，这样 10:00-11:00 与 11:00-12:00 紧密衔接时不会误判冲突）。

> **⚠️ 老陈避坑**：坑点：把比较符号写成 `<=` 会导致两场首尾相接的会议被错误判定为冲突！

### 3. 3. 邮件草拟与外发分离（Draft-First Pattern）

企业办公助手里**永远不要给 Agent 提供“一键直接外发外部邮件”的无保护工具**！标准做法是拆成两步：专家 Agent 只能调用 `email_draft_tool` 生成草稿并标记 `HIGH_RISK`，待用户在审批界面确认收件人和正文后，再由后端确定性代码执行实际发送。

> **⚠️ 老陈避坑**：坑点：大模型偶尔会把“抄送给王总”幻觉成 `wang@gmail.com` 等外部错误邮箱，必须用域名/格式正则校验并展示给人工二次核对。


## 💻 实操步骤 1：步骤一：实现 5 大企业办公工具集与统一工作区底座

- **目标文件**：`versions/day48/src/day48_office_tools.py`
- **老陈代码旁白**：我们一口气把项目三的 5 大核心企业工具全部高质量实现出来：日程冲突检测与预定、邮件草拟与邮箱校验、行业资讯搜索、企业文档 RAG 检索、以及请假/报销审批单管理，全部挂载在可隔离的 `OfficeWorkspaceStore` 上！


In [ ]:
"""Day 48 - 步骤一：阶段项目三 5 大企业办公工具集与统一工作区底座。"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Any, Dict, List

EMAIL_RE = re.compile(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")


@dataclass
class ToolResult:
    ok: bool
    tool_name: str
    risk_level: str  # "LOW" | "HIGH_RISK"
    data: Dict[str, Any] = field(default_factory=dict)
    message: str = ""


class OfficeWorkspaceStore:
    """可隔离的企业办公内存工作区（日程/邮件草稿/知识文档/外部资讯/审批单）。"""

    def __init__(self) -> None:
        self.meetings: List[Dict[str, Any]] = [
            {
                "meeting_id": "MTG-001",
                "title": "星辰智脑架构周会",
                "date": "2026-11-20",
                "start_hour": 10,
                "end_hour": 11,
                "attendees": ["老陈", "阿杰", "林产品"],
            }
        ]
        self.email_drafts: Dict[str, Dict[str, Any]] = {}
        self.approvals: Dict[str, Dict[str, Any]] = {}
        self.audit_logs: List[Dict[str, Any]] = []
        self.internal_docs: List[Dict[str, str]] = [
            {
                "doc_id": "DOC-TRAVEL-2026",
                "title": "2026年星辰数科差旅报销管理制度",
                "content": "一线城市住宿上限 600 元/晚，餐补 120 元/天；单笔报销超过 5000 元须经部门总监人工复核。",
            },
            {
                "doc_id": "DOC-LEAVE-2026",
                "title": "员工年假与病假审批规范",
                "content": "年假 3 天及以内由直属主管审批，超过 3 天须部门负责人审批；需提前至少 1 个工作日提交 OA 审批单。",
            },
            {
                "doc_id": "DOC-SECURITY-2026",
                "title": "AI 办公助手数据安全红线",
                "content": "禁止对外发邮件中包含未脱敏客户手机号与内部密钥；高危写操作必须经过 Human-in-the-loop 人工确认。",
            },
        ]
        self.web_news: List[Dict[str, str]] = [
            {
                "title": "2026 企业级多 Agent 协同落地白皮书发布",
                "snippet": "报告指出采用 LangGraph 状态机与 MCP 协议的企业，AI 办公自动化交付周期缩短 45%。",
            },
            {
                "title": "金融与政企加速推进 Text-to-SQL 只读安全问数",
                "snippet": "双重只读防护与语义指标宽表成为企业智能问数标配架构。",
            },
        ]

    def log_audit(self, tool_name: str, actor: str, status: str, detail: str) -> None:
        self.audit_logs.append(
            {"tool": tool_name, "actor": actor, "status": status, "detail": detail}
        )


class EnterpriseOfficeToolkit:
    """封装 5 大企业办公工具：日程、邮件草拟、资讯搜索、文档 RAG、审批单。"""

    def __init__(self, store: OfficeWorkspaceStore) -> None:
        self.store = store

    # 工具 1：日程冲突检测与会议预定
    def calendar_tool(
        self,
        title: str,
        date: str,
        start_hour: int,
        end_hour: int,
        attendees: List[str],
        actor: str = "employee",
    ) -> ToolResult:
        if start_hour >= end_hour or not (0 <= start_hour <= 23 and 1 <= end_hour <= 24):
            self.store.log_audit("calendar_tool", actor, "REJECTED", "非法会议时段")
            return ToolResult(False, "calendar_tool", "LOW", {}, "会议起止时间不合法，要求 start_hour < end_hour")

        for m in self.store.meetings:
            if m["date"] == date and (start_hour < m["end_hour"] and m["start_hour"] < end_hour):
                self.store.log_audit("calendar_tool", actor, "CONFLICT", f"与 {m['meeting_id']} 冲突")
                return ToolResult(
                    False,
                    "calendar_tool",
                    "LOW",
                    {"conflict_with": m},
                    f"日程冲突：{date} {start_hour}:00-{end_hour}:00 与《{m['title']}》({m['start_hour']}:00-{m['end_hour']}:00) 重叠",
                )

        mtg_id = f"MTG-{len(self.store.meetings) + 1:03d}"
        record = {
            "meeting_id": mtg_id,
            "title": title,
            "date": date,
            "start_hour": start_hour,
            "end_hour": end_hour,
            "attendees": attendees,
        }
        self.store.meetings.append(record)
        self.store.log_audit("calendar_tool", actor, "OK", f"已预定 {mtg_id}")
        return ToolResult(True, "calendar_tool", "LOW", record, f"会议《{title}》预定成功（编号 {mtg_id}）")

    # 工具 2：邮件草拟与收件人校验（标记为 HIGH_RISK，需人工确认外发）
    def email_draft_tool(
        self,
        to_email: str,
        subject: str,
        body: str,
        actor: str = "employee",
    ) -> ToolResult:
        if not EMAIL_RE.match(to_email.strip()):
            self.store.log_audit("email_draft_tool", actor, "REJECTED", f"非法邮箱 {to_email}")
            return ToolResult(False, "email_draft_tool", "HIGH_RISK", {}, f"收件人邮箱格式非法: {to_email}")

        draft_id = f"MAIL-{len(self.store.email_drafts) + 1:03d}"
        draft = {
            "draft_id": draft_id,
            "to": to_email.strip(),
            "subject": subject.strip(),
            "body": body.strip(),
            "status": "PENDING_CONFIRMATION",
        }
        self.store.email_drafts[draft_id] = draft
        self.store.log_audit("email_draft_tool", actor, "DRAFTED", f"草稿 {draft_id} 待人工确认")
        return ToolResult(
            True,
            "email_draft_tool",
            "HIGH_RISK",
            draft,
            f"已生成待确认邮件草稿 {draft_id}（收件人: {draft['to']}），等待人工确认后外发",
        )

    # 工具 3：行业资讯模拟网络搜索
    def web_search_tool(self, query: str, actor: str = "employee") -> ToolResult:
        q_chars = {c for c in query.lower() if c.strip()}
        hits = []
        for item in self.store.web_news:
            text = (item["title"] + item["snippet"]).lower()
            score = sum(1 for c in q_chars if c in text)
            if score > 0:
                hits.append((score, item))
        hits.sort(key=lambda x: x[0], reverse=True)
        matched = [x[1] for x in hits[:2]]
        self.store.log_audit("web_search_tool", actor, "OK", f"命中 {len(matched)} 条资讯")
        return ToolResult(True, "web_search_tool", "LOW", {"results": matched}, f"检索到 {len(matched)} 条行业资讯")

    # 工具 4：内部企业文档 RAG 检索
    def doc_rag_tool(self, query: str, actor: str = "employee") -> ToolResult:
        q_chars = {c for c in query.lower() if c.strip()}
        scored = []
        for doc in self.store.internal_docs:
            text = (doc["title"] + doc["content"]).lower()
            overlap = sum(1 for c in q_chars if c in text)
            if overlap > 0:
                scored.append((overlap, doc))
        scored.sort(key=lambda x: x[0], reverse=True)
        top_docs = [x[1] for x in scored[:2]]
        self.store.log_audit("doc_rag_tool", actor, "OK", f"召回 {len(top_docs)} 篇制度文档")
        return ToolResult(True, "doc_rag_tool", "LOW", {"docs": top_docs}, f"召回 {len(top_docs)} 篇企业制度文档")

    # 工具 5：费用/请假审批单创建
    def approval_tool(
        self,
        approval_type: str,  # "expense" | "leave"
        applicant: str,
        amount_or_days: float,
        reason: str,
    ) -> ToolResult:
        if approval_type not in ("expense", "leave") or amount_or_days <= 0:
            return ToolResult(False, "approval_tool", "HIGH_RISK", {}, "审批类型必须为 expense/leave 且数值大于 0")

        is_high_risk = (approval_type == "expense" and amount_or_days > 5000) or (
            approval_type == "leave" and amount_or_days > 3
        )
        risk = "HIGH_RISK" if is_high_risk else "LOW"
        app_id = f"OA-{len(self.store.approvals) + 1:03d}"
        record = {
            "approval_id": app_id,
            "type": approval_type,
            "applicant": applicant,
            "value": amount_or_days,
            "reason": reason,
            "status": "PENDING_HUMAN_APPROVAL" if is_high_risk else "AUTO_SUBMITTED",
        }
        self.store.approvals[app_id] = record
        self.store.log_audit("approval_tool", applicant, record["status"], f"{app_id} ({risk})")
        return ToolResult(
            True,
            "approval_tool",
            risk,
            record,
            f"审批单 {app_id} 已创建（风险等级: {risk}，状态: {record['status']}）",
        )


if __name__ == "__main__":
    store = OfficeWorkspaceStore()
    kit = EnterpriseOfficeToolkit(store)
    print(kit.doc_rag_tool("差旅住宿报销标准"))
    print(kit.calendar_tool("项目三评审", "2026-11-20", 14, 15, ["老陈", "林产品"]))
    print(kit.email_draft_tool("partner@client.com", "合作方案确认", "您好，附件为星辰智脑方案。"))
    print(kit.approval_tool("expense", "阿杰", 6800.0, "华东客户现场实施差旅费"))


### 🔍 步骤 1 代码深度解析

1. 五大工具全部返回标准 `ToolResult`，清楚标明 `ok` 与 `risk_level`（`LOW` / `HIGH_RISK`）。
2. 每次工具调用都会自动写入 `store.audit_logs`，满足企业内控审计要求。
3. `calendar_tool` 使用严格半开区间重叠公式，`email_draft_tool` 采用 Draft-First 模式，`approval_tool` 对大额报销（>5000）与长假（>3天）自动升级为 `HIGH_RISK`。

**预期运行输出参考：**
```text
ToolResult(ok=True, tool_name='doc_rag_tool', risk_level='LOW', ...)
ToolResult(ok=True, tool_name='calendar_tool', risk_level='LOW', ..., message='会议《项目三评审》预定成功（编号 MTG-002）')
ToolResult(ok=True, tool_name='email_draft_tool', risk_level='HIGH_RISK', ..., message='已生成待确认邮件草稿 MAIL-001（收件人: partner@client.com），等待人工确认后外发')
ToolResult(ok=True, tool_name='approval_tool', risk_level='HIGH_RISK', ..., message='审批单 OA-001 已创建（风险等级: HIGH_RISK，状态: PENDING_HUMAN_APPROVAL）')
```


## 💻 实操步骤 2：步骤二：实现三大领域专家 Agent（知识/日程邮件/审批）与权限边界

- **目标文件**：`versions/day48/src/day48_office_agents.py`
- **老陈代码旁白**：工具齐全了，现在把它们按最小权限原则封装进三位领域专家 Agent：`KnowledgeAgent`、`SchedulerAgent`、`ApprovalAgent`。每位专家只绑定自己专属的工具，对外暴露统一的 `handle(request, context)` 接口，随时准备接入明天 Day 49 的 LangGraph 编排器！


In [ ]:
"""Day 48 - 步骤二：多 Agent 智能办公助手三大领域专家 Agent。"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List

from src.day48_office_tools import EnterpriseOfficeToolkit, OfficeWorkspaceStore, ToolResult


@dataclass
class AgentResponse:
    agent_name: str
    summary: str
    requires_human_confirmation: bool
    tool_results: List[ToolResult]


class KnowledgeAgent:
    """知识检索专员：仅拥有只读权限（doc_rag_tool + web_search_tool）。"""

    name = "knowledge_agent"
    allowed_tools = ["doc_rag_tool", "web_search_tool"]

    def __init__(self, toolkit: EnterpriseOfficeToolkit) -> None:
        self.toolkit = toolkit

    def handle(self, query: str, params: Dict[str, Any] | None = None) -> AgentResponse:
        results: List[ToolResult] = []
        if any(k in query for k in ("行业", "趋势", "资讯", "白皮书", "新闻")):
            r = self.toolkit.web_search_tool(query)
            results.append(r)
            snippets = [x["snippet"] for x in r.data.get("results", [])]
            summary = "；".join(snippets) or "未检索到相关行业资讯。"
        else:
            r = self.toolkit.doc_rag_tool(query)
            results.append(r)
            docs = r.data.get("docs", [])
            summary = "；".join(f"《{d['title']}》:{d['content']}" for d in docs) or "未检索到相关内部制度。"
        return AgentResponse(self.name, summary, False, results)


class SchedulerAgent:
    """日程与邮件专员：绑定 calendar_tool 与 email_draft_tool。"""

    name = "scheduler_agent"
    allowed_tools = ["calendar_tool", "email_draft_tool"]

    def __init__(self, toolkit: EnterpriseOfficeToolkit) -> None:
        self.toolkit = toolkit

    def handle(self, query: str, params: Dict[str, Any] | None = None) -> AgentResponse:
        p = params or {}
        if "邮件" in query or "email" in query.lower():
            res = self.toolkit.email_draft_tool(
                to_email=p.get("to_email", "client@example.com"),
                subject=p.get("subject", "星辰数科项目同步"),
                body=p.get("body", query),
                actor=p.get("actor", "employee"),
            )
            return AgentResponse(
                self.name,
                res.message,
                requires_human_confirmation=(res.ok and res.risk_level == "HIGH_RISK"),
                tool_results=[res],
            )

        res = self.toolkit.calendar_tool(
            title=p.get("title", "项目沟通会"),
            date=p.get("date", "2026-11-20"),
            start_hour=int(p.get("start_hour", 14)),
            end_hour=int(p.get("end_hour", 15)),
            attendees=list(p.get("attendees", ["老陈", "林产品"])),
            actor=p.get("actor", "employee"),
        )
        return AgentResponse(self.name, res.message, False, [res])


class ApprovalAgent:
    """审批流程专员：绑定 approval_tool，大额报销/长假自动要求人工确认。"""

    name = "approval_agent"
    allowed_tools = ["approval_tool"]

    def __init__(self, toolkit: EnterpriseOfficeToolkit) -> None:
        self.toolkit = toolkit

    def handle(self, query: str, params: Dict[str, Any] | None = None) -> AgentResponse:
        p = params or {}
        app_type = p.get("approval_type", "leave" if "请假" in query or "年假" in query else "expense")
        value = float(p.get("value", 1000.0 if app_type == "expense" else 1.0))
        res = self.toolkit.approval_tool(
            approval_type=app_type,
            applicant=p.get("actor", "employee"),
            amount_or_days=value,
            reason=p.get("reason", query),
        )
        return AgentResponse(
            self.name,
            res.message,
            requires_human_confirmation=(res.ok and res.risk_level == "HIGH_RISK"),
            tool_results=[res],
        )


if __name__ == "__main__":
    store = OfficeWorkspaceStore()
    kit = EnterpriseOfficeToolkit(store)
    ka = KnowledgeAgent(kit)
    sa = SchedulerAgent(kit)
    aa = ApprovalAgent(kit)

    print("知识专员:", ka.handle("差旅报销制度"))
    print("日程专员:", sa.handle("帮我发邮件给客户", {"to_email": "boss@partner.cn", "subject": "报价确认"}))
    print("审批专员:", aa.handle("提交 8000 元差旅报销", {"approval_type": "expense", "value": 8000.0}))


### 🔍 步骤 2 代码深度解析

1. 三位专家通过 `allowed_tools` 明确最小权限边界，`KnowledgeAgent` 永远不会触发任何写操作。
2. 统一返回 `AgentResponse`，其中 `requires_human_confirmation` 字段直接把高危动作信号暴露给明天 Day 49 的 LangGraph 编排器触发 `INTERRUPTED` 中断。

**预期运行输出参考：**
```text
知识专员: AgentResponse(agent_name='knowledge_agent', ..., requires_human_confirmation=False, ...)
日程专员: AgentResponse(agent_name='scheduler_agent', ..., requires_human_confirmation=True, ...)
审批专员: AgentResponse(agent_name='approval_agent', ..., requires_human_confirmation=True, ...)
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day48/tests/test_day48.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 48 自动化单元测试：验证 5 大企业办公工具边界与三大领域专家 Agent。"""

from src.day48_office_tools import EnterpriseOfficeToolkit, OfficeWorkspaceStore
from src.day48_office_agents import ApprovalAgent, KnowledgeAgent, SchedulerAgent


def test_five_office_tools_and_boundaries():
    store = OfficeWorkspaceStore()
    kit = EnterpriseOfficeToolkit(store)

    # 1. 日程冲突拦截与空闲预定
    conflict = kit.calendar_tool("冲突会议", "2026-11-20", 10, 11, ["老陈"])
    assert conflict.ok is False and "冲突" in conflict.message
    ok_mtg = kit.calendar_tool("相邻会议", "2026-11-20", 11, 12, ["老陈"])
    assert ok_mtg.ok is True and ok_mtg.data["meeting_id"] == "MTG-002"

    # 2. 邮件格式拦截与高危草稿
    bad_mail = kit.email_draft_tool("invalid-email", "标题", "正文")
    assert bad_mail.ok is False
    ok_mail = kit.email_draft_tool("cto@partner.com", "合作方案", "请查收")
    assert ok_mail.ok is True and ok_mail.risk_level == "HIGH_RISK"

    # 3. 资讯搜索与文档 RAG
    news = kit.web_search_tool("MCP 协议多 Agent")
    assert news.ok is True and len(news.data["results"]) >= 1
    docs = kit.doc_rag_tool("差旅住宿报销")
    assert docs.ok is True and "600 元/晚" in docs.data["docs"][0]["content"]

    # 4. 审批单风险分级
    small_exp = kit.approval_tool("expense", "阿杰", 800.0, "打车费")
    assert small_exp.risk_level == "LOW"
    big_exp = kit.approval_tool("expense", "阿杰", 6500.0, "服务器采购")
    assert big_exp.risk_level == "HIGH_RISK"
    assert len(store.audit_logs) >= 6


def test_three_specialist_office_agents():
    store = OfficeWorkspaceStore()
    kit = EnterpriseOfficeToolkit(store)

    k_resp = KnowledgeAgent(kit).handle("员工年假规范")
    assert k_resp.requires_human_confirmation is False
    assert "3 天" in k_resp.summary

    s_resp = SchedulerAgent(kit).handle("发邮件通知客户", {"to_email": "ops@client.com", "subject": "周报"})
    assert s_resp.requires_human_confirmation is True

    a_resp = ApprovalAgent(kit).handle("申请 5 天年假", {"approval_type": "leave", "value": 5})
    assert a_resp.requires_human_confirmation is True

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：为日程工具增加空闲时段自动推荐功能

**题目要求：**
当目标会议时间冲突时，编写 `suggest_free_slots(meetings, date, duration=1)`，在当天 09:00~18:00 内找出所有不冲突的空闲 1 小时时段。

**思路提示：** 遍历 `hour in range(9, 18)`，检查 `[hour, hour + duration)` 是否与当天所有已有会议不重叠。


In [ ]:
"""Day 48 作业 1 参考答案：工作日空闲会议时段自动推荐器。"""

from typing import Dict, List, Tuple


def suggest_free_slots(meetings: List[Dict[str, int]], duration: int = 1) -> List[Tuple[int, int]]:
    free: List[Tuple[int, int]] = []
    for h in range(9, 18 - duration + 1):
        end = h + duration
        conflict = any(h < m["end_hour"] and m["start_hour"] < end for m in meetings)
        if not conflict:
            free.append((h, end))
    return free


if __name__ == "__main__":
    booked = [{"start_hour": 10, "end_hour": 11}, {"start_hour": 14, "end_hour": 16}]
    slots = suggest_free_slots(booked, duration=1)
    print("推荐空闲时段:", slots)
    assert (10, 11) not in slots and (11, 12) in slots and (14, 15) not in slots


**答案解析与评分标准：**

从“单纯拒绝冲突”升级到“拒绝 + 主动推荐替代空闲时段”，是优秀办公 Agent 体验的分水岭。


### 作业 2（企业实战）：为邮件工具增加外部域名告警检测

**题目要求：**
编写 `is_external_email(email, internal_domain='starmind.cn')`，判断收件人是否属于外部域名，若是则在草稿中附加 `external_warning=True` 提示人工加倍谨慎。

**思路提示：** 按 `@` 分割邮箱取域名部分，转为小写后与 `internal_domain` 比较。


In [ ]:
"""Day 48 作业 2 参考答案：企业外发邮件外部域名检测器。"""


def is_external_email(email: str, internal_domain: str = "starmind.cn") -> bool:
    domain = email.strip().lower().split("@")[-1]
    return domain != internal_domain.lower()


if __name__ == "__main__":
    assert is_external_email("chen@starmind.cn") is False
    assert is_external_email("ceo@partner.com") is True
    print("外部域名检测通过")


**答案解析与评分标准：**

区分内部邮件与外部客户邮件，可以让安全策略更精细（例如内部提醒低风险，外发客户强拦截）。


### 作业 3（架构进阶）：将审计日志导出为按工具维度统计的调用报表

**题目要求：**
编写 `summarize_audit_logs(logs)`，统计每个工具的调用总次数与被拦截/冲突次数，为后续运维看板提供数据源。

**思路提示：** 使用 `collections.defaultdict` 按 `log['tool']` 聚合统计 `total` 与 `blocked`。


In [ ]:
"""Day 48 作业 3 参考答案：企业办公工具审计日志聚合统计。"""

from collections import defaultdict
from typing import Dict, List


def summarize_audit_logs(logs: List[Dict[str, str]]) -> Dict[str, Dict[str, int]]:
    stats: Dict[str, Dict[str, int]] = defaultdict(lambda: {"total": 0, "blocked": 0})
    for entry in logs:
        t = entry["tool"]
        stats[t]["total"] += 1
        if entry["status"] in ("REJECTED", "CONFLICT"):
            stats[t]["blocked"] += 1
    return dict(stats)


if __name__ == "__main__":
    sample = [
        {"tool": "calendar_tool", "status": "CONFLICT"},
        {"tool": "calendar_tool", "status": "OK"},
        {"tool": "email_draft_tool", "status": "REJECTED"},
    ]
    report = summarize_audit_logs(sample)
    print("审计汇总:", report)
    assert report["calendar_tool"] == {"total": 2, "blocked": 1}


**答案解析与评分标准：**

高冲突率或高拦截率的工具往往暴露了 Prompt 引导不足或前端表单校验缺失，审计聚合报表能直接指导迭代。


---

## 🎯 老陈收尾复盘

漂亮！今天项目三的地基打得稳如泰山：5 大企业办公工具全部具备统一 `ToolResult` 契约、参数校验和审计日志；3 名专家 Agent 职责清晰、权限严格隔离，而且提前标明了哪些操作属于 `HIGH_RISK`（如发邮件、超额报销审批）。明天 Day 49，我们把它们接入 LangGraph 编排器 + HITL 人工审批断点 + FastAPI 接口，彻底让整个智能办公助手活起来！
